# VLM-счётчик на тесте и train: шарды 0 и 1 из 4
Вход: `ts-code`, данные соревнования (или `ts-data-private`), веса Qwen3-VL-4B-Instruct. GPU T4 ×2, интернет выключен. Запуск: Save Version (Run All). Каждый шард идёт своим процессом на своей GPU.

Выход: `/kaggle/working/artifacts/rowfeats/_parts/vlm_qwen__v1/shard_<i>_of_4.jsonl`. Сборка всех четырёх шардов: `python tracks/det/vlm/run_vlm.py --subset all --merge`.

Возобновление: подключите вывод прошлой версии и скопируйте его `artifacts/rowfeats/_parts` в `/kaggle/working/artifacts/rowfeats/_parts` до запуска.

In [ ]:
# Подготовка VLM-прогона: код из датасета ts-code, данные соревнования, веса Qwen-VL. Интернет выключен.
import os, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
code = [p.parent for p in INPUT.rglob('run.py') if (p.parent / 'ts' / '__init__.py').is_file()]
data = [p.parent for p in INPUT.rglob('train_labels.csv')
        if (p.parent / 'tables.csv').is_file() and (p.parent / 'data_card.json').is_file()]
qwen = sorted(p.parent for p in INPUT.rglob('config.json') if 'qwen' in str(p).lower())
assert len(code) == 1, f'Подключите датасет ts-code (найдено {len(code)})'
assert len(data) == 1, f'Подключите данные соревнования или их копию (найдено {len(data)})'
assert qwen, 'Подключите веса Qwen3-VL-4B-Instruct (Kaggle Models qwen-lm/qwen-3-vl, вариант 4b-instruct)'
REPO, QWEN = Path('/kaggle/working/table-state'), qwen[0]
shutil.copytree(code[0], REPO, dirs_exist_ok=True)
os.environ['TS_DATA'] = str(data[0])
os.environ['TS_ARTIFACTS'] = '/kaggle/working/artifacts'
os.chdir(REPO)
import torch, transformers
print('код:', (REPO / 'GIT_SHA').read_text().strip(), '| данные:', data[0], '| веса VLM:', QWEN)
print('transformers', transformers.__version__, '| torch', torch.__version__, '| GPU:', torch.cuda.device_count())


In [ ]:
!python tracks/det/vlm/run_vlm.py --model {QWEN} --subset all --shard 0 --of 4 --device cuda:0 --prompt-id vlm_v1 > /kaggle/working/shard0.log 2>&1 & python tracks/det/vlm/run_vlm.py --model {QWEN} --subset all --shard 1 --of 4 --device cuda:1 --prompt-id vlm_v1; wait; tail -3 /kaggle/working/shard0.log

In [ ]:
!wc -l /kaggle/working/artifacts/rowfeats/_parts/vlm_qwen__v1/*.jsonl; rm -rf /kaggle/working/table-state